# Fetch NPTEL Transcripts (run in Google Colab)

Run this notebook in Colab to bypass YouTube IP rate limits.
After running all cells, download `transcripts.zip` from the Files panel.
Then unzip into `data/lecture_rag_75/transcripts/` in your local project.

In [ ]:
!pip install youtube-transcript-api -q

In [ ]:
import os, json, time
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound

# All 16 video IDs across 4 domains
DOMAINS = {
    "machine_learning": [
        "fC7V8QsPBec",  # NPTEL ML by Ravindran, IIT Madras
        "OTAR0kT1swg",
        "HTSCbxSxs-g",
        "9vMpHk44XXo",
    ],
    "computer_networks": [
        "UXMIxCYZu8o",  # NPTEL CN by Sujoy Ghosh, IIT KGP
        "5D67Qy1tPLY",
        "HExZQZFkVY4",
        "AYdF7b3nMto",
    ],
    "database_systems": [
        "OWX4RvijwLw",  # NPTEL DBMS by Partha Pratim Das, IIT KGP
        "rrG7azSlyWI",
        "s1Jb-NJNpT4",
        "rbwXdTsCk2c",
    ],
    "operating_systems": [
        "cngXL3_Pros",  # NPTEL OS by Chester Rebeiro, IIT Madras
        "YGe9e9jiK8A",
        "xSfW2SSN0jM",
        "Se1iohX6N9A",
    ],
}

DELAY = 2.0   # seconds between requests
OUT_DIR = "/content/transcripts"
os.makedirs(OUT_DIR, exist_ok=True)

api = YouTubeTranscriptApi()

results = {}
for domain, video_ids in DOMAINS.items():
    domain_dir = os.path.join(OUT_DIR, domain)
    os.makedirs(domain_dir, exist_ok=True)
    domain_texts = []
    for vid in video_ids:
        print(f"  [{domain}] {vid} ...", end=" ", flush=True)
        for attempt in range(3):
            try:
                transcript = api.fetch(vid, languages=["en", "en-US", "en-GB"])
                segments = [(seg.start, seg.text) for seg in transcript]
                text = " ".join(s[1] for s in segments)
                domain_texts.append(text)
                # Save per-video
                with open(os.path.join(domain_dir, f"{vid}.txt"), "w", encoding="utf-8") as f:
                    for start, txt in segments:
                        f.write(f"[{start:.1f}] {txt}\n")
                print(f"OK ({len(segments)} segs)")
                break
            except (TranscriptsDisabled, NoTranscriptFound) as e:
                print(f"No captions: {e}")
                break
            except Exception as e:
                if attempt < 2:
                    time.sleep(DELAY * (attempt + 1))
                else:
                    print(f"FAILED: {e}")
        time.sleep(DELAY)
    
    # Save merged domain transcript
    if domain_texts:
        merged = " ".join(domain_texts)
        merged_path = os.path.join(domain_dir, f"{domain}.txt")
        with open(merged_path, "w", encoding="utf-8") as f:
            f.write(merged)
        results[domain] = {"videos": len(domain_texts), "chars": len(merged)}
        print(f"  -> Merged {len(domain_texts)} videos -> {merged_path} ({len(merged)} chars)")
    else:
        results[domain] = {"videos": 0, "chars": 0}
        print(f"  -> No transcripts for {domain}")

print("\nSummary:", json.dumps(results, indent=2))

In [ ]:
# Zip and download
import shutil
shutil.make_archive("/content/transcripts", "zip", "/content", "transcripts")
print("Created /content/transcripts.zip")
print("Download it from the Files panel (folder icon on the left)")
print("")
print("After downloading, extract to your project:")
print("  Unzip transcripts.zip")
print("  Copy transcripts/<domain>/<domain>.txt -> data/lecture_rag_75/transcripts/<domain>/<domain>.txt")